In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import os
import math
from pathlib import Path
 
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)

In [3]:
DATA_DIR    = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
MODEL_NAME  = "microsoft/deberta-v3-base"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN     = 256
BATCH_SIZE  = 4
EPOCHS      = 3
LR          = 2e-5        
WARMUP_FRAC = 0.2      
GRAD_CLIP   = 0.5        
USE_FP16    = DEVICE == "cuda"  
OPTION_COLS = ["A", "B", "C", "D", "E"]
 
print(f"Device : {DEVICE}")
print(f"FP16   : {USE_FP16}")
print(f"Model  : {MODEL_NAME}")

Device : cuda
FP16   : True
Model  : microsoft/deberta-v3-base


In [4]:
def apk(actual, predicted, k=3):
    if not actual:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)
 
def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
 
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
 
print(f"Train: {len(train_df)} rows  |  Test: {len(test_df)} rows")

Train: 2000 rows  |  Test: 500 rows


In [6]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.has_labels = has_labels
        self.label_map  = {c: i for i, c in enumerate(OPTION_COLS)}
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        question = str(row["prompt"])
        choices  = [str(row[c]) for c in OPTION_COLS]
 
        enc = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v for k, v in enc.items()}
 
        if self.has_labels:
            item["labels"] = torch.tensor(
                self.label_map[str(row["answer"])], dtype=torch.long
            )
        return item

In [7]:
print("\nLoading tokenizer and model …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
 
# ★ Cast model to float32 explicitly (prevents silent fp16 init issues)
model = model.float()
 
train_dataset = MCQDataset(train_df, tokenizer, has_labels=True)
test_dataset  = MCQDataset(test_df,  tokenizer, has_labels=False)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
 


Loading tokenizer and model …


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                  

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [8]:
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped = [
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
]
optimizer   = AdamW(optimizer_grouped, lr=LR, eps=1e-6)   # ★ eps=1e-6 more stable than default 1e-8
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16) 

/tmp/ipykernel_23/2759090808.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)


In [9]:
print("\n" + "="*50)
print("Fine-tuning DeBERTa-v3-base …")
print("="*50)
 
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    nan_batches = 0
 
    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)
 
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
 
        # ★ NaN guard — skip bad batch instead of poisoning weights
        if not math.isfinite(loss.item()):
            nan_batches += 1
            optimizer.zero_grad()
            continue
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
 
        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
 
        if (step + 1) % 50 == 0:
            avg = total_loss / max(step + 1 - nan_batches, 1)
            print(f"  step {step+1}/{len(train_loader)}  "
                  f"loss={avg:.4f}  acc={correct/total:.4f}  "
                  f"nan_skipped={nan_batches}")
 
    epoch_loss = total_loss / max(len(train_loader) - nan_batches, 1)
    epoch_acc  = correct / max(total, 1)
    print(f"\nEpoch {epoch+1}/{EPOCHS} — "
          f"loss: {epoch_loss:.4f}  acc: {epoch_acc:.4f}  "
          f"nan_batches_skipped: {nan_batches}\n")
 


Fine-tuning DeBERTa-v3-base …


/tmp/ipykernel_23/1685899588.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):
/tmp/ipykernel_23/1685899588.py:34: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  step 50/500  loss=1.6103  acc=0.1900  nan_skipped=0
  step 100/500  loss=1.6120  acc=0.1875  nan_skipped=0
  step 150/500  loss=1.6111  acc=0.2033  nan_skipped=0
  step 200/500  loss=1.6135  acc=0.2062  nan_skipped=0
  step 250/500  loss=1.6124  acc=0.2080  nan_skipped=0
  step 300/500  loss=1.6127  acc=0.2042  nan_skipped=0
  step 350/500  loss=1.6125  acc=0.2036  nan_skipped=0
  step 400/500  loss=1.6122  acc=0.2056  nan_skipped=0
  step 450/500  loss=1.6119  acc=0.2094  nan_skipped=0
  step 500/500  loss=1.6110  acc=0.2130  nan_skipped=0

Epoch 1/3 — loss: 1.6110  acc: 0.2130  nan_batches_skipped: 0

  step 50/500  loss=1.6242  acc=0.1900  nan_skipped=0
  step 100/500  loss=1.6212  acc=0.1925  nan_skipped=0
  step 150/500  loss=1.6155  acc=0.2050  nan_skipped=0
  step 200/500  loss=1.6182  acc=0.2200  nan_skipped=0
  step 250/500  loss=1.6147  acc=0.2200  nan_skipped=0
  step 300/500  loss=1.6131  acc=0.2283  nan_skipped=0
  step 350/500  loss=1.6153  acc=0.2229  nan_skipped=0
  s

In [10]:
def predict_top3(loader):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs.logits.float().cpu().numpy())
    logits = np.vstack(all_logits)
    preds  = []
    for row in logits:
        ranked = [OPTION_COLS[i] for i in np.argsort(row)[::-1]]
        preds.append(ranked[:3])
    return logits, preds

In [11]:
print("Evaluating on training set …")
_, train_preds = predict_top3(DataLoader(train_dataset, batch_size=BATCH_SIZE))
train_map3     = mapk(train_df["answer"].tolist(), train_preds)
print(f"Train MAP@3: {train_map3:.4f}")
 
print("\nGenerating test predictions …")
_, test_preds = predict_top3(test_loader)
 
submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
out_path = "/kaggle/working/submission_deberta.csv"
submission.to_csv(out_path, index=False)
print(f"\n  Saved → {out_path}")
print(submission.head(10).to_string(index=False))

Evaluating on training set …


/tmp/ipykernel_23/1236647199.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):


Train MAP@3: 0.3701

Generating test predictions …

  Saved → /kaggle/working/submission_deberta.csv
 ID Prediction
  1      C E D
  2      E D C
  3      E C A
  4      E D C
  5      B A C
  6      E B D
  7      E D B
  8      A E D
  9      C D E
 10      D E C
